# 6차 — 기상예보 기반 풍력발전량 예측 (연구 노트북)

**목표**: 리더보드 산식 `0.5×(1-NMAE) + 0.5×FICR` 최대화.

**5차 대비 변경**: 보정·피처·가중치는 5차 그대로 두고, **모델 다양성**만 추가 — LightGBM 단독 → **LightGBM + XGBoost + CatBoost 3중 등가중 블렌드**.

## 설계 원칙 (검증으로 확정된 것들)

1. **평가시간 가중치**(실발전 ≥ 용량 10% 시간대만 weight 1) — 다년도에서 최적. 에너지비례·지수 가중치는 NMAE 손실이 FICR 이득을 초과해 기각.
2. **그룹별 보정**: g1 scale↑(과소편향), g2·g3 stretch 제거(과대편향). LB에서 확인됨(g3 +0.010, g1/g2 +0.001).
3. **모델 다양성**: CatBoost(ordered boosting)·XGBoost는 LGB와 오차 상관이 낮음. exp13 다년도 검증에서 3중 블렌드가 LGB 단독 대비 2023 +0.0027 / 2024 +0.0007 (모든 그룹 개선). (참고: HistGBM은 LGB와 유사해 +0.0004로 무의미했음.)

## 연구 로그 (확정 LB)

| 제출 | 변경 | LB |
|---|---|---|
| 4차-v1 | 초기(공격적 보정) | 0.6202 |
| 4차-v2 | g3 tail-stretch 제거 | 0.6302 (+0.010) |
| **5차** | g1/g2 보정 다년도 재튜닝 | **0.6312** (FICR 0.3991) |
| **6차** | +LGB/XGB/CatBoost 3중 블렌드 | 확인 예정 (로컬 다년도 +0.002~0.003) |

> **핵심 교훈**: 단일 유리한 홀드아웃(2024)은 과신을 부른다. 실제 LB는 어려운 2023 홀드아웃과 일치. 모든 선택은 **2023+2024 다년도(특히 어려운 해)** 기준으로 판단한다.

## 규칙 준수 (Data Leakage)

- 기상예보는 제공된 `data_available_kst_dtm`(발표시각) 그대로 사용 — 예측기준시점 이후 자료 없음.
- **시간 파생 피처는 issue-safe**: lag는 과거 발표분(안전), lead는 같은 발표 블록 내부로 제한(경계 넘으면 NaN).
- SCADA(실측)는 **학습 구간 라벨 정제에만** 사용 (테스트 피처 미사용).
- XGBoost·CatBoost는 오픈소스(Apache-2.0)·로컬 실행 → 규칙 준수. 외부 데이터·API·사전학습 가중치 미사용. 본 노트북만으로 재현 가능.

## 1. Setup — 상수 · 평가 산식

대회 공식 산식을 그대로 구현합니다. `metric_group`은 그룹 단위 축약판으로,
3그룹 평균과 동치이므로 그룹별 독립 튜닝이 전체 점수 최적화와 일치합니다.

In [1]:
import numpy as np, pandas as pd, time, warnings
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.isotonic import IsotonicRegression
warnings.filterwarnings("ignore")
t_start = time.time()

D = "data/"
TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
HUB = 117.0            # 허브 높이(m) — info.xlsx
GROUP_CENTROID = {     # info.xlsx 터빈 좌표의 그룹별 중심
    "kpx_group_1": (37.28713, 128.95202),
    "kpx_group_2": (37.28225, 128.96515),
    "kpx_group_3": (37.27520, 128.97144),
}

def metric_group(actual, forecast, capacity):
    """대회 산식(그룹 단위): 실발전 >= 용량 10% 시간대만, 0.5*(1-NMAE)+0.5*FICR."""
    valid = actual >= capacity * 0.10
    a, f = actual[valid], forecast[valid]
    er = np.abs(f - a) / capacity
    nmae = er.mean()
    price = np.select([er <= 0.06, er <= 0.08], [4.0, 3.0], default=0.0)
    ficr = (a * price).sum() / (a * 4.0).sum()
    return 0.5 * (1 - nmae) + 0.5 * ficr, nmae, ficr

## 2. 피처 엔지니어링

**물리 기반 파생** — 발전량 P ∝ ρ·v³ 이므로:
- u/v 합성 풍속(`hypot`), 윈드시어 멱법칙으로 **허브고도(117m) 풍속 외삽**, 세제곱항, 공기밀도(p/RT)
- LDAPS 50m max/min 차이 → 돌풍(gust) 대리변수

**공간** — 격자 16(LDAPS)/9(GFS)개를 (a) 전체 평균/최대/표준편차(공유), (b) **그룹 중심점 IDW 가중합성**(그룹 전용)으로 요약.

**시간(issue-safe)** — 같은 발표 블록(`data_available_kst_dtm` 동일) 안에서만 이웃시각 참조:
- lag(1~3h): 과거 발표분 포함 → 무조건 안전
- lead(1~3h): 블록 경계를 넘으면 미래 발표분 → **NaN 마스킹**
- 발표 블록 통계(그날 풍속 프로파일: 평균/최대/표준편차/편차) — 전부 발표시점에 알려진 값
- `lead_h`: 예보 경과시간(12~35h) — 예보 열화 정도

In [2]:
def _ldaps_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    u10, v10 = df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"]
    o["ws10"] = np.hypot(u10, v10)
    u50 = (df["heightAboveGround_50_50MUmax"] + df["heightAboveGround_50_50MUmin"]) / 2
    v50 = (df["heightAboveGround_50_50MVmax"] + df["heightAboveGround_50_50MVmin"]) / 2
    o["ws50"] = np.hypot(u50, v50)
    o["ws50max"] = np.hypot(df["heightAboveGround_50_50MUmax"], df["heightAboveGround_50_50MVmax"])
    o["ws50_gust"] = (df["heightAboveGround_50_50MUmax"] - df["heightAboveGround_50_50MUmin"]).abs() \
        + (df["heightAboveGround_50_50MVmax"] - df["heightAboveGround_50_50MVmin"]).abs()
    o["blws"] = np.hypot(df["heightAboveGround_5_XBLWS"], df["heightAboveGround_5_YBLWS"])
    alpha = np.log(np.clip(o["ws50"], 0.1, None) / np.clip(o["ws10"], 0.1, None)) / np.log(50 / 10)
    o["shear_a"] = alpha = alpha.clip(-0.5, 1.0)
    o["ws_hub"] = o["ws50"] * (HUB / 50.0) ** alpha       # 멱법칙 허브고도 외삽
    o["ws_hub3"] = o["ws_hub"] ** 3                        # P ∝ v³
    o["ws50_3"] = o["ws50"] ** 3
    o["rho"] = df["surface_0_sp"] / (287.05 * df["heightAboveGround_2_t"])   # 이상기체
    o["t2"] = df["heightAboveGround_2_t"]; o["sp"] = df["surface_0_sp"]; o["blh"] = df["etc_0_blh"]
    o["u50"], o["v50"] = u50, v50
    return o

def _gfs_rowfeats(df):
    o = pd.DataFrame(index=df.index)
    o["gws10"] = np.hypot(df["heightAboveGround_10_10u"], df["heightAboveGround_10_10v"])
    o["gws80"] = np.hypot(df["heightAboveGround_80_u"], df["heightAboveGround_80_v"])
    o["gws100"] = np.hypot(df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"])
    o["gws100_3"] = o["gws100"] ** 3
    o["gust"] = df["surface_0_gust"]
    o["gpbl"] = np.hypot(df["planetaryBoundaryLayer_0_u"], df["planetaryBoundaryLayer_0_v"])
    o["gws850"] = np.hypot(df["isobaricInhPa_850_u"], df["isobaricInhPa_850_v"])
    o["gws700"] = np.hypot(df["isobaricInhPa_700_u"], df["isobaricInhPa_700_v"])
    o["gu100"], o["gv100"] = df["heightAboveGround_100_100u"], df["heightAboveGround_100_100v"]
    return o

def calendar_features(idx):
    dt = pd.to_datetime(pd.Series(idx))
    out = pd.DataFrame(index=idx)
    h, m, doy = dt.dt.hour.values, dt.dt.month.values, dt.dt.dayofyear.values
    out["hour_sin"] = np.sin(2*np.pi*h/24);  out["hour_cos"] = np.cos(2*np.pi*h/24)
    out["month_sin"] = np.sin(2*np.pi*m/12); out["month_cos"] = np.cos(2*np.pi*m/12)
    out["doy_sin"] = np.sin(2*np.pi*doy/365); out["doy_cos"] = np.cos(2*np.pi*doy/365)
    return out

In [3]:
def _grid_coords(df):
    return df.drop_duplicates("grid_id").set_index("grid_id")[["latitude", "longitude"]].sort_index()

def _idw_weights(gridc, clat, clon, power=2.0):
    dy = (gridc["latitude"] - clat) * 111.0
    dx = (gridc["longitude"] - clon) * 111.0 * np.cos(np.radians(clat))
    w = (1.0 / (dx**2 + dy**2 + 0.25)) ** (power / 2.0)
    return w / w.sum()

def _time_pivot(rowfeats, times, gridids, col):
    rf = pd.DataFrame({"t": times.values, "g": gridids.values, "v": rowfeats[col].values})
    return rf.pivot_table(index="t", columns="g", values="v")

def _idw_series(pivot, weights):
    w = weights.reindex(pivot.columns).fillna(0.0).values
    return pd.Series(pivot.values @ w, index=pivot.index)

def build_shared(ldaps, gfs):
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)
    def agg(rf, times, prefix):
        x = rf.copy(); x["t"] = times.values; g = x.groupby("t")
        m = g.mean(); m.columns = [f"{prefix}_{c}_mean" for c in m.columns]
        mx = g.max(); mx = mx[[c for c in mx.columns if c != "t"]]
        mx.columns = [f"{prefix}_{c}_max" for c in mx.columns]
        sd = g.std(); sd.columns = [f"{prefix}_{c}_std" for c in sd.columns]
        keep_mx = [c for c in mx.columns if "ws" in c or "gust" in c]
        keep_sd = [c for c in sd.columns if "ws" in c]
        return pd.concat([m, mx[keep_mx], sd[keep_sd]], axis=1)
    feat = agg(lrf, lt, "l").join(agg(grf, gt, "g"), how="left")
    av = ldaps.groupby(ldaps["forecast_kst_dtm"])["data_available_kst_dtm"].first()
    av.index = pd.to_datetime(av.index)
    feat.index = pd.to_datetime(feat.index)
    feat["_avail"] = pd.to_datetime(av.reindex(feat.index).values)   # 발표 블록 id
    return feat.join(calendar_features(feat.index), how="left")

def build_group_extra(ldaps, gfs):
    lt = pd.to_datetime(ldaps["forecast_kst_dtm"]); gt = pd.to_datetime(gfs["forecast_kst_dtm"])
    lrf = _ldaps_rowfeats(ldaps); grf = _gfs_rowfeats(gfs)
    lgc = _grid_coords(ldaps); ggc = _grid_coords(gfs)
    piv = {c: _time_pivot(lrf, lt, ldaps["grid_id"], c) for c in
           ["ws_hub", "ws_hub3", "ws50", "u50", "v50", "ws50_gust", "shear_a", "rho"]}
    gpiv = {c: _time_pivot(grf, gt, gfs["grid_id"], c) for c in ["gws100", "gust"]}
    out = {}
    for tgt, (clat, clon) in GROUP_CENTROID.items():
        lw = _idw_weights(lgc, clat, clon); gw = _idw_weights(ggc, clat, clon)
        e = pd.DataFrame(index=piv["ws_hub"].index)
        e["gws_hub"] = _idw_series(piv["ws_hub"], lw)
        e["gws_hub3"] = _idw_series(piv["ws_hub3"], lw)
        e["gws50"] = _idw_series(piv["ws50"], lw)
        e["ggust"] = _idw_series(piv["ws50_gust"], lw)
        e["gshear"] = _idw_series(piv["shear_a"], lw)
        e["grho"] = _idw_series(piv["rho"], lw)
        u = _idw_series(piv["u50"], lw); v = _idw_series(piv["v50"], lw)
        ang = np.arctan2(v, u)
        e["gwd_sin"] = np.sin(ang); e["gwd_cos"] = np.cos(ang)
        e["gws_x_sin"] = e["gws50"] * e["gwd_sin"]; e["gws_x_cos"] = e["gws50"] * e["gwd_cos"]
        e["ggfs100"] = _idw_series(gpiv["gws100"], gw)
        e["ggfs_gust"] = _idw_series(gpiv["gust"], gw)
        e["l_g_ratio"] = e["gws_hub"] / np.clip(e["ggfs100"], 0.1, None)
        e.index = pd.to_datetime(e.index)
        out[tgt] = e
    return out

TEMPORAL_BASE = ["gws_hub", "gws50", "ggfs100", "ggust"]

def add_temporal(X, avail):
    """issue-safe 시간 피처: lag는 자유, lead는 같은 발표 블록 내부만(경계 넘으면 NaN)."""
    X = X.sort_index(); out = X.copy()
    blk = pd.Series(avail.values, index=X.index)
    for c in TEMPORAL_BASE:
        s = X[c]
        for k in [1, 2, 3]:
            out[f"{c}_lag{k}"] = s.shift(k)
            ok = blk.shift(-k) == blk
            out[f"{c}_lead{k}"] = s.shift(-k).where(ok.values)
        out[f"{c}_d1"] = s - s.shift(1)
        out[f"{c}_roll3m"] = s.rolling(3, min_periods=1).mean()
        out[f"{c}_roll6m"] = s.rolling(6, min_periods=1).mean()
        out[f"{c}_roll6s"] = s.rolling(6, min_periods=2).std()
    g = pd.Series(X["gws_hub"].values, index=X.index).groupby(blk.values)
    out["day_ws_mean"] = g.transform("mean").values
    out["day_ws_max"] = g.transform("max").values
    out["day_ws_std"] = g.transform("std").values
    out["ws_vs_day"] = out["gws_hub"] - out["day_ws_mean"]
    out["lead_h"] = (X.index - pd.DatetimeIndex(avail.values)).total_seconds() / 3600.0
    return out

def group_matrix(tgt, shared, extra):
    return add_temporal(shared.drop(columns=["_avail"]).join(extra[tgt], how="left"), shared["_avail"])

## 3. 데이터 로드 · SCADA 라벨 디노이징

**디노이징 근거**: 출력제한(curtailment)·정지 시간대는 "바람→발전량" 관계 밖의 라벨 노이즈.
SCADA 풍속으로 isotonic 경험 파워커브를 적합하고, 실발전이 커브 대비 **용량의 12% 이상 미달**한 시각을 학습에서 제외
(2024 홀드아웃에서 +0.010 개선 확인). SCADA는 여기(학습 라벨 정제)에만 쓰이고 테스트 피처로는 사용하지 않습니다.

In [4]:
ldaps = pd.read_csv(D+"train/ldaps_train.csv"); gfs = pd.read_csv(D+"train/gfs_train.csv")
ldaps_te = pd.read_csv(D+"test/ldaps_test.csv"); gfs_te = pd.read_csv(D+"test/gfs_test.csv")
lab = pd.read_csv(D+"train/train_labels.csv")
lab.columns = [c.strip("﻿") for c in lab.columns]
lab["kst_dtm"] = pd.to_datetime(lab["kst_dtm"]); lab = lab.set_index("kst_dtm")
sub = pd.read_csv(D+"sample_submission.csv")
sub.columns = [c.strip("﻿") for c in sub.columns]
sub["forecast_kst_dtm"] = pd.to_datetime(sub["forecast_kst_dtm"])

shared = build_shared(ldaps, gfs);       extra = build_group_extra(ldaps, gfs)
shared_te = build_shared(ldaps_te, gfs_te); extra_te = build_group_extra(ldaps_te, gfs_te)
print(f"features built  train={shared.shape} test={shared_te.shape}  {time.time()-t_start:.0f}s")

features built  train=(26304, 61) test=(8760, 61)  4s


In [5]:
DENOISE_THRESH = -0.12   # 파워커브 대비 -12%p 초과 미달 시각 제거

def scada_group_ws(path, prefix, cols):
    df = pd.read_csv(path); df["kst_dtm"] = pd.to_datetime(df["kst_dtm"])
    df["hour"] = df["kst_dtm"].dt.floor("h")
    ws = [f"{prefix}_wtg{i:02d}_ws" for i in cols]
    for c in ws: df[c] = df[c].clip(0, 40)
    g = df.groupby("hour"); n = g.size()
    return pd.DataFrame({"scada_ws": g[ws].mean().mean(axis=1), "n": n}).query("n==6")

SCADA_WS = {"kpx_group_1": scada_group_ws(D+"train/scada_vestas_train.csv", "vestas", range(1, 7)),
            "kpx_group_2": scada_group_ws(D+"train/scada_vestas_train.csv", "vestas", range(7, 13)),
            "kpx_group_3": scada_group_ws(D+"train/scada_unison_train.csv", "unison", range(1, 6))}
BAD_HOURS = {}
for tgt in TARGET_COLS:
    j = lab[[tgt]].join(SCADA_WS[tgt][["scada_ws"]]).dropna()
    ir = IsotonicRegression(out_of_bounds="clip").fit(j["scada_ws"], j[tgt])   # 경험 파워커브
    resid = (j[tgt] - ir.predict(j["scada_ws"])) / CAPACITY_KWH[tgt]
    BAD_HOURS[tgt] = j.index[(resid < DENOISE_THRESH).values]
    print(f"{tgt}: 제거 {len(BAD_HOURS[tgt])}시간 ({len(BAD_HOURS[tgt])/len(j)*100:.1f}%)")

kpx_group_1: 제거 2384시간 (9.1%)
kpx_group_2: 제거 2137시간 (8.2%)
kpx_group_3: 제거 1547시간 (8.8%)


## 4. 모델 레시피

- **3중 모델 다양성**: LightGBM + XGBoost + CatBoost, 각각 quantile α∈{0.50,0.55,0.60} 앙상블 → **모델별 CF를 등가중 평균**. (LGB는 시드[42,7,2024] 추가 앙상블; XGB/Cat은 다양성 목적이라 단일 시드.)
- **per-group + pooled 블렌딩**: 각 모델 내부에서 그룹별 모델과 3그룹 통합 모델(원-핫)을 그룹가중 `w`로 혼합. g3는 데이터 적어 pooled 비중↑(w=0.25).
- **평가시간 가중치**: 실발전 ≥ 용량 10% 시간대만 weight 1 (다년도에서 에너지비례/지수 가중치보다 우수).
- **보정(2단, 그룹별)**: `CF×scale` 후 상단확장 `cf>hi_piv면 hi_piv+(cf-hi_piv)·k_hi`, `[0,용량]` 클리핑. 5차 설정 그대로.

| 그룹 | blend w | scale | hi_piv | k_hi |
|---|---|---|---|---|
| 1 | 0.375 | 1.050 | 0.60 | 1.15 |
| 2 | 0.625 | 1.035 | 0.70 | 1.00 |
| 3 | 0.250 | 1.030 | 0.70 | 1.00 |

> **6차 근거(exp13, 다년도)**: LGB+XGB+Cat 3중 블렌드가 LGB 단독 대비 2023 +0.0027 / 2024 +0.0007, 모든 그룹 개선. 보정은 5차와 동일(다양성과 직교).

In [6]:
ALPHAS = [0.50, 0.55, 0.60]
LGB_SEEDS = [42, 7, 2024]     # LGB는 시드 앙상블(분산↓), XGB/Cat은 다양성 목적이라 단일 시드
XGB_SEEDS = [42]
CAT_SEEDS = [42]
GID = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
# 6차: 5차 보정 그대로 + 모델 다양성(LGB+XGB+CatBoost 3중 등가중 블렌드).
#  exp13 다년도 검증: 3중 블렌드가 LGB 단독 대비 2023 +0.0027 / 2024 +0.0007 (모든 그룹 개선).
#  CatBoost(ordered boosting)·XGBoost는 LGB와 오차 상관이 낮아 진짜 다양성 이득(이전 HGB는 +0.0004로 무의미했음).
CFG = {"kpx_group_1": dict(w=0.375, scale=1.050, hi_piv=0.60, k_hi=1.15),
       "kpx_group_2": dict(w=0.625, scale=1.035, hi_piv=0.70, k_hi=1.00),
       "kpx_group_3": dict(w=0.250, scale=1.030, hi_piv=0.70, k_hi=1.00)}

def lgbm_kw(a, s):
    return dict(objective="quantile", alpha=a, n_estimators=1200, learning_rate=0.03,
        num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=1.0, random_state=s, n_jobs=-1, verbose=-1)

def make_model(name, a, s):
    if name == "lgb":
        return lgb.LGBMRegressor(**lgbm_kw(a, s))
    if name == "xgb":
        return xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=a, n_estimators=1200,
            learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            min_child_weight=20, random_state=s, n_jobs=-1)
    return CatBoostRegressor(loss_function=f"Quantile:alpha={a}", n_estimators=1200, learning_rate=0.03,
        depth=6, l2_leaf_reg=3.0, subsample=0.8, random_seed=s, thread_count=-1, verbose=0,
        allow_writing_files=False)

MODELS = [("lgb", LGB_SEEDS, False), ("xgb", XGB_SEEDS, True), ("cat", CAT_SEEDS, True)]  # 3번째=NaN 대체 필요

def apply_calib(cf, cap, c):
    cf2 = cf * c["scale"]
    m = cf2 > c["hi_piv"]
    cf2[m] = c["hi_piv"] + (cf2[m] - c["hi_piv"]) * c["k_hi"]
    return np.clip(cf2 * cap, 0, cap)

def fit_predict(train_end, pred_index_per_group, shared_tr, extra_tr, shared_pr, extra_pr):
    """LGB+XGB+CatBoost 3중 등가중 블렌드. 각 모델은 per-group+pooled를 그룹가중 w로 혼합."""
    # 그룹별 학습/예측 행렬 준비 (디노이즈 + 평가시간 가중치)
    Xtr_g, Xpr_g, ytr_g, wtr_g = {}, {}, {}, {}
    for tgt in TARGET_COLS:
        X = group_matrix(tgt, shared_tr, extra_tr); Xp = group_matrix(tgt, shared_pr, extra_pr)
        y = lab[tgt]; idx = lab.index[y.notna() & (lab.index <= train_end)].difference(BAD_HOURS[tgt])
        cap = CAPACITY_KWH[tgt]; ykwh = y.loc[idx].values
        Xtr_g[tgt] = X.reindex(idx); ytr_g[tgt] = ykwh / cap
        wtr_g[tgt] = (ykwh >= 0.1 * cap).astype(float)
        Xpr_g[tgt] = Xp.reindex(pred_index_per_group[tgt])

    def onehot(df, tgt):
        o = df.copy()
        for g in range(3): o[f"g{g+1}"] = int(GID[tgt] == g)
        return o
    Xtr_pool = pd.concat([onehot(Xtr_g[t], t) for t in TARGET_COLS])
    ytr_pool = np.concatenate([ytr_g[t] for t in TARGET_COLS])
    wtr_pool = np.concatenate([wtr_g[t] for t in TARGET_COLS])
    Xpr_pool = {t: onehot(Xpr_g[t], t) for t in TARGET_COLS}

    model_cfs = {t: [] for t in TARGET_COLS}
    for name, seeds, needfill in MODELS:
        prep = (lambda df: df.fillna(-999)) if needfill else (lambda df: df)
        # per-group
        per = {}
        for tgt in TARGET_COLS:
            Xt, Xp = prep(Xtr_g[tgt]), prep(Xpr_g[tgt])
            ps = [np.clip(make_model(name, a, s).fit(Xt, ytr_g[tgt], sample_weight=wtr_g[tgt]).predict(Xp), 0, 1)
                  for a in ALPHAS for s in seeds]
            per[tgt] = np.mean(ps, axis=0)
        # pooled
        Xtp = prep(Xtr_pool); pool = {t: [] for t in TARGET_COLS}
        for a in ALPHAS:
            for s in seeds:
                mdl = make_model(name, a, s).fit(Xtp, ytr_pool, sample_weight=wtr_pool)
                for tgt in TARGET_COLS:
                    pool[tgt].append(np.clip(mdl.predict(prep(Xpr_pool[tgt])), 0, 1))
        for tgt in TARGET_COLS:
            w = CFG[tgt]["w"]
            model_cfs[tgt].append(w * per[tgt] + (1 - w) * np.mean(pool[tgt], axis=0))
        print(f"  [{name}] done {time.time()-t_start:.0f}s")
    return {t: np.mean(model_cfs[t], axis=0) for t in TARGET_COLS}   # 3중 등가중

## 5. 검증 — 2024년 홀드아웃

학습 ≤2023 / 검증 2024 전체. 최종 레시피 그대로 실행해 대회 산식으로 채점합니다.
(그룹3은 검증 시 학습 데이터가 2023년 1년뿐 — 최종 모델은 2023-24 2년을 쓰므로 보수적 추정치)

In [7]:
VAL_START = pd.Timestamp("2024-01-01 01:00:00")
val_idx = {t: lab.index[(lab.index >= VAL_START) & lab[t].notna()] for t in TARGET_COLS}
cf_val = fit_predict(VAL_START - pd.Timedelta(hours=1), val_idx, shared, extra, shared, extra)

scores = []
for tgt in TARGET_COLS:
    cap = CAPACITY_KWH[tgt]
    pred = apply_calib(cf_val[tgt].copy(), cap, CFG[tgt])
    s, nmae, ficr = metric_group(lab.loc[val_idx[tgt], tgt].values, pred, cap)
    scores.append(s)
    print(f"{tgt}: score={s:.4f}  1-NMAE={1-nmae:.4f}  FICR={ficr:.4f}")
print(f"\nTOTAL (2024 holdout) = {np.mean(scores):.4f}   {time.time()-t_start:.0f}s")

  [lgb] done 556s


  [xgb] done 661s


  [cat] done 750s
kpx_group_1: score=0.6579  1-NMAE=0.8801  FICR=0.4356
kpx_group_2: score=0.6653  1-NMAE=0.8747  FICR=0.4559
kpx_group_3: score=0.5977  1-NMAE=0.8665  FICR=0.3288

TOTAL (2024 holdout) = 0.6403   750s


## 6. 최종 학습 · 제출 파일 생성

전체 학습 데이터(2022-2024, 그룹3은 2023-2024)로 재학습하고 2025년 전체(8,760시간)를 예측합니다.

In [8]:
test_idx = {t: pd.DatetimeIndex(sub["forecast_kst_dtm"]) for t in TARGET_COLS}
cf_test = fit_predict(lab.index.max(), test_idx, shared, extra, shared_te, extra_te)

out = sub[["forecast_id", "forecast_kst_dtm"]].copy()
for tgt in TARGET_COLS:
    out[tgt] = apply_calib(cf_test[tgt].copy(), CAPACITY_KWH[tgt], CFG[tgt])
out["forecast_kst_dtm"] = out["forecast_kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")
out.to_csv(D+"submission_6차.csv", index=False, encoding="utf-8-sig")
print(f"saved {D}submission_6차.csv   {time.time()-t_start:.0f}s")
out.head()

  [lgb] done 1263s


  [xgb] done 1397s


  [cat] done 1509s
saved data/submission_6차.csv   1509s


,forecast_id,forecast_kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,forecast_0001,2025-01-01 01:00:00,19980.222801,19034.937592,17321.293338
1,forecast_0002,2025-01-01 02:00:00,20062.678962,19230.110762,17340.966424
2,forecast_0003,2025-01-01 03:00:00,19774.161793,19283.216117,17355.668892
3,forecast_0004,2025-01-01 04:00:00,19450.259715,19038.489676,16894.113045
4,forecast_0005,2025-01-01 05:00:00,19569.052570,19171.833934,16995.095116


## 7. 무결성 점검

제출 형식·값 범위·계절 분포가 정상인지 확인합니다.

In [9]:
chk = pd.read_csv(D+"submission_6차.csv")
assert list(chk.columns) == list(sub.columns), "컬럼 불일치"
assert len(chk) == len(sub) == 8760, "행 수 불일치"
assert chk[TARGET_COLS].notna().all().all(), "NaN 존재"
for t in TARGET_COLS:
    assert (chk[t] >= 0).all() and (chk[t] <= CAPACITY_KWH[t]).all(), f"{t} 범위 초과"
mon = chk.assign(m=pd.to_datetime(chk["forecast_kst_dtm"]).dt.month).groupby("m")[TARGET_COLS].mean()
tr_mon = lab.assign(m=lab.index.month).groupby("m")[TARGET_COLS].mean()
print("월별 평균 발전량 — 예측(2025) vs 학습(2022-24):")
print(pd.concat({"pred": mon.round(0), "train": tr_mon.round(0)}, axis=1).to_string())
print("\n모든 점검 통과 ✓")

월별 평균 발전량 — 예측(2025) vs 학습(2022-24):
          pred                               train                        
   kpx_group_1 kpx_group_2 kpx_group_3 kpx_group_1 kpx_group_2 kpx_group_3
m                                                                         
1      12724.0     12727.0     10659.0      9355.0      9685.0      8499.0
2      13679.0     13716.0     11969.0      6845.0      6980.0      4593.0
3       9300.0      9734.0      8217.0      7099.0      7682.0      6601.0
4       9989.0     10385.0      8899.0      6909.0      7510.0      5484.0
5       8364.0      8509.0      7434.0      7053.0      7259.0      5450.0
6       7777.0      8092.0      6845.0      4931.0      5315.0      3594.0
7       6639.0      7031.0      5864.0      5638.0      6208.0      7521.0
8       6775.0      7113.0      6044.0      3975.0      4758.0      2490.0
9       5077.0      5530.0      4302.0      3184.0      3612.0      2047.0
10      4191.0      4457.0      3357.0      4831.0      5040.0 